# Adjacency matrix

In [ ]:
from lingam import DirectLiNGAM

model = DirectLiNGAM()
model.fit(reduced_data)

# Extract adjacency matrix
adj_matrix = model.adjacency_matrix_

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 124 iterations, alpha=3.115e-05, previous alpha=2.302e-05, with an active set of 61 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 90 iterations, alpha=4.604e-07, previous alpha=4.543e-07, with an active set of 63 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 96 iterations, alpha=7.951e-07, previous alpha=7.861e-07, with an active set of 63 regressors.
  warnings.warn(
/usr/l

In [ ]:
import networkx as nx

# Create a directed graph
causal_graph = nx.DiGraph()
num_features = reduced_data.shape[1]

# Add edges based on LiNGAM's adjacency matrix
for i in range(num_features):
  for j in range(num_features):
    if adj_matrix[i, j] != 0:  # Nonzero values indicate causal relationships
      causal_graph.add_edge(i, j, weight=adj_matrix[i, j])

In [ ]:
# Create a DataFrame for the reduced data with original column names
reduced_data_df = pd.DataFrame(data=reduced_data, index=norm_df.index, columns=[norm_df.columns[i] for i in range(100)])

# CausalNN

In [ ]:
# =============================
# Step 2: Define the CausalNN Model
# =============================
class CausalNN(Model):
    def __init__(self, input_dim, latent_dim, causal_graph):
        """
        CausalNN for dimensionality reduction, integrating a causal graph.

        :param input_dim: Number of input features.
        :param latent_dim: Size of the reduced representation.
        :param causal_graph: NetworkX graph defining causal feature relationships.
        """
        super(CausalNN, self).__init__()
        self.latent_dim = latent_dim
        self.causal_graph = causal_graph  # Store learned causal structure

        # Encoder
        self.encoder = tf.keras.Sequential([
            layers.InputLayer(input_shape=(input_dim,)),
            layers.Dense(128, activation='leaky_relu', activity_regularizer=tf.keras.regularizers.l1(1e-5)),
            layers.Dropout(0.2),  # Prevents redundancy
            layers.Dense(64, activation='leaky_relu'),
            layers.Dense(latent_dim, activation='tanh')  # Tanh captures non-linear interactions
        ])

        # Decoder
        self.decoder = tf.keras.Sequential([
            layers.InputLayer(input_shape=(latent_dim,)),
            layers.Dense(64, activation='leaky_relu'),
            layers.Dense(128, activation='leaky_relu'),
            layers.Dense(input_dim, activation=None)  # Matches z-score normalization
        ])

    def call(self, inputs):
        encoded = self.encoder(inputs)
        decoded = self.decoder(encoded)
        return decoded

    def compute_causal_penalty(self, inputs):
        """
        Computes a causal loss to enforce learned representations to respect the causal structure.
        """
        adjacency_matrix = nx.to_numpy_array(self.causal_graph)
        adjacency_matrix = tf.convert_to_tensor(adjacency_matrix, dtype=tf.float32)

        # Get the encoded feature representation
        encoded = self.encoder(inputs)

        # Select only nodes that exist in the causal graph
        encoded_subset = tf.gather(encoded, indices=list(self.causal_graph.nodes), axis=-1)

        # Reshape for proper matrix multiplication
        encoded_subset = tf.reshape(encoded_subset, [tf.shape(encoded_subset)[0], len(self.causal_graph.nodes)])

        # Enforce causal consistency by penalizing violations in causal structure
        causal_penalty = tf.reduce_sum(tf.abs(tf.matmul(encoded_subset, adjacency_matrix)))

        return causal_penalty

# =============================
# Step 3: Define the Loss Function
# =============================
def compute_loss(model, x, reconstructed):
    mse_loss = tf.keras.losses.MeanSquaredError()(x, reconstructed)
    mae_loss = tf.keras.losses.MeanAbsoluteError()(x, reconstructed)
    causal_loss = model.compute_causal_penalty(x) * 0.01  # Small weight for causal regularization
    return 0.5 * mse_loss + 0.5 * mae_loss + causal_loss

# =============================
# Step 4: Training Function
# =============================
def train_causal_nn(data, input_dim, latent_dim, causal_graph, epochs=50, batch_size=64):
    """
    Trains the CausalNN model.

    :param data: NumPy array of omics data.
    :param input_dim: Number of input features.
    :param latent_dim: Dimension of reduced feature space.
    :param causal_graph: Causal structure learned from LiNGAM.
    :param epochs: Number of training epochs.
    :param batch_size: Batch size for optimization.
    :return: Trained CausalNN model.
    """
    model = CausalNN(input_dim, latent_dim, causal_graph)
    optimizer = tf.keras.optimizers.Adam()

    for epoch in range(epochs):
        for batch_start in range(0, data.shape[0], batch_size):
            batch_data = data[batch_start:batch_start + batch_size]
            with tf.GradientTape() as tape:
                reconstructed = model(batch_data)
                loss = compute_loss(model, batch_data, reconstructed)
            gradients = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        print(f"Epoch {epoch + 1}, Loss: {loss.numpy():.4f}")

    return model

In [ ]:
# =============================
# Step 5: Run the Pipeline
# =============================
if __name__ == "__main__":
    import pandas as pd

    # Load omics dataset (assumed to be preprocessed)
    data_array = reduced_data_df.values.astype(np.float32)

    # Define model parameters
    input_dim = data_array.shape[1]
    latent_dim = 100
    epochs = 50
    batch_size = 64

    # Train the CausalNN model
    causal_nn = train_causal_nn(data_array, input_dim, latent_dim, causal_graph, epochs, batch_size)

    # Encode the data into the latent space
    reduced_shape = causal_nn.encoder(data_array).numpy()

    print("Reduced data shape:", reduced_shape.shape)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1, Loss: 43.4667
Epoch 2, Loss: 33.8088
Epoch 3, Loss: 27.3120
Epoch 4, Loss: 22.3797
Epoch 5, Loss: 19.1031
Epoch 6, Loss: 16.6846
Epoch 7, Loss: 16.7720
Epoch 8, Loss: 16.4148
Epoch 9, Loss: 14.0190
Epoch 10, Loss: 14.3736
Epoch 11, Loss: 13.5092
Epoch 12, Loss: 12.1788
Epoch 13, Loss: 12.1548
Epoch 14, Loss: 11.6645
Epoch 15, Loss: 10.4345
Epoch 16, Loss: 10.6917
Epoch 17, Loss: 10.0003
Epoch 18, Loss: 9.4146
Epoch 19, Loss: 8.9082
Epoch 20, Loss: 9.0969
Epoch 21, Loss: 7.9353
Epoch 22, Loss: 8.1016
Epoch 23, Loss: 7.7273
Epoch 24, Loss: 7.6332
Epoch 25, Loss: 7.4096
Epoch 26, Loss: 7.1702
Epoch 27, Loss: 7.9791
Epoch 28, Loss: 6.7907
Epoch 29, Loss: 7.4961
Epoch 30, Loss: 6.8586
Epoch 31, Loss: 6.2809
Epoch 32, Loss: 6.2147
Epoch 33, Loss: 5.8942
Epoch 34, Loss: 5.5693
Epoch 35, Loss: 5.7291
Epoch 36, Loss: 5.3245
Epoch 37, Loss: 5.5513
Epoch 38, Loss: 5.3193
Epoch 39, Loss: 5.2603
Epoch 40, Loss: 5.1155
Epoch 41, Loss: 4.8516
Epoch 42, Loss: 4.9767
Epoch 43, Loss: 4.9403
Epo

In [ ]:
model1 = DirectLiNGAM()
model1.fit(reduced_data1)

# Extract adjacency matrix
adj_matrix1 = model1.adjacency_matrix_

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 34 iterations, alpha=1.337e-04, previous alpha=1.336e-04, with an active set of 11 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 23 iterations, alpha=3.822e-07, previous alpha=3.344e-07, with an active set of 14 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 52 iterations, alpha=6.178e-07, previous alpha=5.205e-07, with an active set of 17 regressors.
  warnings.warn(
/usr/lo

In [ ]:
# Create a directed graph
causal_graph1 = nx.DiGraph()
num_features = reduced_data1.shape[1]

# Add edges based on LiNGAM's adjacency matrix
for i in range(num_features):
  for j in range(num_features):
    if adj_matrix1[i, j] != 0:  # Nonzero values indicate causal relationships
      causal_graph1.add_edge(i, j, weight=adj_matrix1[i, j])

In [ ]:
# Create a DataFrame for the reduced data with original column names
reduced_data_df1 = pd.DataFrame(data=reduced_data1, index=norm_df1.index, columns=[norm_df1.columns[i] for i in range(100)])

In [ ]:
# =============================
# Step 5: Run the Pipeline
# =============================
if __name__ == "__main__":
    import pandas as pd

    # Load omics dataset (assumed to be preprocessed)
    data_array1 = reduced_data_df1.values.astype(np.float32)

    # Define model parameters
    input_dim = data_array1.shape[1]
    latent_dim = 100
    epochs = 50
    batch_size = 64

    # Train the CausalNN model
    causal_nn = train_causal_nn(data_array1, input_dim, latent_dim, causal_graph1, epochs, batch_size)

    # Encode the data into the latent space
    reduced_shape1 = causal_nn.encoder(data_array1).numpy()

    print("Reduced data shape:", reduced_shape1.shape)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1, Loss: 20.5289
Epoch 2, Loss: 11.7752
Epoch 3, Loss: 7.0799
Epoch 4, Loss: 4.1401
Epoch 5, Loss: 2.8499
Epoch 6, Loss: 2.3579
Epoch 7, Loss: 2.6301
Epoch 8, Loss: 2.2560
Epoch 9, Loss: 1.8025
Epoch 10, Loss: 2.0957
Epoch 11, Loss: 1.5137
Epoch 12, Loss: 1.3798
Epoch 13, Loss: 1.7810
Epoch 14, Loss: 1.5293
Epoch 15, Loss: 1.1179
Epoch 16, Loss: 1.4803
Epoch 17, Loss: 1.1100
Epoch 18, Loss: 1.3557
Epoch 19, Loss: 1.3025
Epoch 20, Loss: 0.9646
Epoch 21, Loss: 1.4631
Epoch 22, Loss: 1.3746
Epoch 23, Loss: 0.8928
Epoch 24, Loss: 1.1307
Epoch 25, Loss: 1.1672
Epoch 26, Loss: 1.3806
Epoch 27, Loss: 1.3594
Epoch 28, Loss: 1.0493
Epoch 29, Loss: 1.1451
Epoch 30, Loss: 1.0885
Epoch 31, Loss: 1.0752
Epoch 32, Loss: 1.1622
Epoch 33, Loss: 1.1392
Epoch 34, Loss: 1.2264
Epoch 35, Loss: 1.7011
Epoch 36, Loss: 1.6375
Epoch 37, Loss: 1.2872
Epoch 38, Loss: 1.3008
Epoch 39, Loss: 0.9953
Epoch 40, Loss: 0.8804
Epoch 41, Loss: 1.2594
Epoch 42, Loss: 1.4499
Epoch 43, Loss: 1.3016
Epoch 44, Loss: 0.

In [ ]:
model2 = DirectLiNGAM()
model2.fit(reduced_data2)

# Extract adjacency matrix
adj_matrix2 = model2.adjacency_matrix_

In [ ]:
# Create a directed graph
causal_graph2 = nx.DiGraph()
num_features = reduced_data2.shape[1]

# Add edges based on LiNGAM's adjacency matrix
for i in range(num_features):
  for j in range(num_features):
    if adj_matrix2[i, j] != 0:  # Nonzero values indicate causal relationships
      causal_graph2.add_edge(i, j, weight=adj_matrix2[i, j])

In [ ]:
# Create a DataFrame for the reduced data with original column names
reduced_data_df2 = pd.DataFrame(data=reduced_data2, index=norm_df2.index, columns=[norm_df2.columns[i] for i in range(50)])

In [ ]:
# =============================
# Step 5: Run the Pipeline
# =============================
if __name__ == "__main__":
    import pandas as pd

    # Load omics dataset (assumed to be preprocessed)
    data_array2 = reduced_data_df2.values.astype(np.float32)
    # Define model parameters
    input_dim = data_array2.shape[1]
    latent_dim = 50
    epochs = 50
    batch_size = 64

    # Train the CausalNN model
    causal_nn = train_causal_nn(data_array2, input_dim, latent_dim, causal_graph2, epochs, batch_size)

    # Encode the data into the latent space
    reduced_shape2 = causal_nn.encoder(data_array2).numpy()

    print("Reduced data shape:", reduced_shape2.shape)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1, Loss: 10.1624
Epoch 2, Loss: 7.3420
Epoch 3, Loss: 5.8354
Epoch 4, Loss: 5.0048
Epoch 5, Loss: 4.3839
Epoch 6, Loss: 3.8023
Epoch 7, Loss: 3.3619
Epoch 8, Loss: 3.0082
Epoch 9, Loss: 2.8421
Epoch 10, Loss: 2.7793
Epoch 11, Loss: 2.8721
Epoch 12, Loss: 2.5206
Epoch 13, Loss: 2.3331
Epoch 14, Loss: 2.2968
Epoch 15, Loss: 2.1961
Epoch 16, Loss: 2.0390
Epoch 17, Loss: 1.9256
Epoch 18, Loss: 1.8786
Epoch 19, Loss: 1.8561
Epoch 20, Loss: 1.8187
Epoch 21, Loss: 1.6451
Epoch 22, Loss: 1.6273
Epoch 23, Loss: 1.6057
Epoch 24, Loss: 1.6205
Epoch 25, Loss: 1.5852
Epoch 26, Loss: 1.5154
Epoch 27, Loss: 1.3874
Epoch 28, Loss: 1.4358
Epoch 29, Loss: 1.3842
Epoch 30, Loss: 1.4334
Epoch 31, Loss: 1.3694
Epoch 32, Loss: 1.2839
Epoch 33, Loss: 1.2219
Epoch 34, Loss: 1.2163
Epoch 35, Loss: 1.2215
Epoch 36, Loss: 1.2300
Epoch 37, Loss: 1.1955
Epoch 38, Loss: 1.1796
Epoch 39, Loss: 1.1355
Epoch 40, Loss: 1.1815
Epoch 41, Loss: 1.0727
Epoch 42, Loss: 1.0746
Epoch 43, Loss: 1.0971
Epoch 44, Loss: 1.0

In [ ]:
model3 = DirectLiNGAM()
model3.fit(reduced_data3)

# Extract adjacency matrix
adj_matrix3 = model3.adjacency_matrix_

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 93 iterations, alpha=1.856e-05, previous alpha=1.852e-05, with an active set of 52 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 107 iterations, alpha=2.321e-06, previous alpha=2.291e-06, with an active set of 56 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 147 iterations, alpha=4.727e-05, previous alpha=4.666e-05, with an active set of 52 regressors.
  warnings.warn(
/usr/

In [ ]:
# Create a directed graph
causal_graph3 = nx.DiGraph()
num_features = reduced_data3.shape[1]

# Add edges based on LiNGAM's adjacency matrix
for i in range(num_features):
  for j in range(num_features):
    if adj_matrix3[i, j] != 0:  # Nonzero values indicate causal relationships
      causal_graph3.add_edge(i, j, weight=adj_matrix3[i, j])

In [ ]:
# Create a DataFrame for the reduced data with original column names
reduced_data_df3 = pd.DataFrame(data=reduced_data3, index=norm_df3.index, columns=[norm_df3.columns[i] for i in range(100)])

In [ ]:
# =============================
# Step 5: Run the Pipeline
# =============================
if __name__ == "__main__":
    import pandas as pd

    # Load omics dataset (assumed to be preprocessed)
    data_array3 = reduced_data_df3.values.astype(np.float32)

    # Define model parameters
    input_dim = data_array3.shape[1]
    latent_dim = 100
    epochs = 50
    batch_size = 64

    # Train the CausalNN model
    causal_nn = train_causal_nn(data_array3, input_dim, latent_dim, causal_graph3, epochs, batch_size)

    # Encode the data into the latent space
    reduced_shape3 = causal_nn.encoder(data_array3).numpy()

    print("Reduced data shape:", reduced_shape3.shape)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Epoch 1, Loss: 28.0620
Epoch 2, Loss: 21.2447
Epoch 3, Loss: 15.8481
Epoch 4, Loss: 11.9856
Epoch 5, Loss: 9.0343
Epoch 6, Loss: 7.5142
Epoch 7, Loss: 6.4520
Epoch 8, Loss: 5.6846
Epoch 9, Loss: 5.1685
Epoch 10, Loss: 4.7394
Epoch 11, Loss: 4.4069
Epoch 12, Loss: 4.1274
Epoch 13, Loss: 3.8770
Epoch 14, Loss: 3.6590
Epoch 15, Loss: 3.4729
Epoch 16, Loss: 3.3157
Epoch 17, Loss: 3.1807
Epoch 18, Loss: 3.0718
Epoch 19, Loss: 2.9864
Epoch 20, Loss: 2.8930
Epoch 21, Loss: 2.8213
Epoch 22, Loss: 2.7682
Epoch 23, Loss: 2.7295
Epoch 24, Loss: 2.6446
Epoch 25, Loss: 2.7194
Epoch 26, Loss: 2.5601
Epoch 27, Loss: 2.5900
Epoch 28, Loss: 2.4714
Epoch 29, Loss: 2.5269
Epoch 30, Loss: 2.4631
Epoch 31, Loss: 2.4098
Epoch 32, Loss: 2.2990
Epoch 33, Loss: 2.3034
Epoch 34, Loss: 2.2152
Epoch 35, Loss: 2.1662
Epoch 36, Loss: 2.1400
Epoch 37, Loss: 2.1053
Epoch 38, Loss: 2.0708
Epoch 39, Loss: 2.0662
Epoch 40, Loss: 2.0450
Epoch 41, Loss: 2.0160
Epoch 42, Loss: 1.9789
Epoch 43, Loss: 1.9727
Epoch 44, Loss: 